# Causal Inference ~ Group 10

## Problem 3 ~ RDD Strategy

We further investigate the sharp RDD strategy employed in Lee (2008), but with a different outcome of interest: the total number of votes a candidate receives in the next election (`next_tv`), as opposed to the vote share. The running variable is the candidate's vote share in the prior election (`share`), with the cutoff `c = 0.5`. The treatment variable is incumbency (`inc`), where `inc = 1` if the candidate won the prior election (i.e. `share >= 0.5`).

## Part A ~ Constructing Control Variables

We create two control variables:

- `R_minus_c` = (R - c) = (share - 0.5): the distance between the candidate's prior vote share and the 50% threshold.
- `S_R_minus_c` = S(R - c) = inc * (share - 0.5): the running-variable distance interacted with incumbency status.

These two controls allow the regression line to have different slopes on each side of the cutoff, which is the standard local-linear specification for a sharp RDD.

In [4]:
import pandas as pd
import statsmodels.api as sm

# Load data
df = pd.read_csv("causal_data/clean_lee_data.csv")

# Define cutoff
c = 0.5

# Create the two controls: (R - c) and S(R - c)
df["R_minus_c"] = df["share"] - c
df["S_R_minus_c"] = df["inc"] * df["R_minus_c"]

print("Summary statistics for control variables:\n")
print(df[["share", "inc", "R_minus_c", "S_R_minus_c", "next_tv"]].describe())
print("\nFirst 5 rows of new control variables:")
print(df[["share", "inc", "R_minus_c", "S_R_minus_c"]].head())

# Explicit values referenced in the interpretation below
print("\nKey values referenced in interpretation:")
print(f"Mean of R_minus_c:    {df['R_minus_c'].mean():.4f}")
print(f"Mean of inc:          {df['inc'].mean():.4f}")

Summary statistics for control variables:

             share          inc    R_minus_c  S_R_minus_c       next_tv
count  4746.000000  4746.000000  4746.000000  4746.000000  4.746000e+03
mean      0.552472     0.566582     0.052472     0.103977  8.054543e+04
std       0.187105     0.495599     0.187105     0.134897  3.990584e+04
min       0.025072     0.000000    -0.474928    -0.000000  3.784000e+03
25%       0.408846     0.000000    -0.091154     0.000000  5.529900e+04
50%       0.533739     1.000000     0.033739     0.033739  7.508650e+04
75%       0.679061     1.000000     0.179061     0.179061  9.996300e+04
max       1.000000     1.000000     0.500000     0.500000  1.436831e+06

First 5 rows of new control variables:
      share  inc  R_minus_c  S_R_minus_c
0  0.469256    0  -0.030744    -0.000000
1  0.553026    1   0.053026     0.053026
2  0.569626    1   0.069626     0.069626
3  0.463041    0  -0.036959    -0.000000
4  0.543411    1   0.043411     0.043411

Key values referenced 

## Part A Interpretation
The constructed `R_minus_c` variable centers the running variable at the 0.5 threshold, with negative values for losers and positive values for winners. Its mean (**0.0525**) is slightly above zero, consistent with the slight tilt toward incumbents (`inc` mean = **0.5666**) in the sample. The interaction `S_R_minus_c` is mechanically zero whenever `inc = 0` and equal to `R_minus_c` when `inc = 1`; this is what allows the RDD specification to fit a separate slope above the cutoff. With these two controls in the regression, the coefficient on `inc` will be identified as the jump in expected `next_tv` exactly at the threshold, purged of the underlying linear relationship between vote share and votes received.

## Part B ~ Estimating the Causal Effect of Incumbency on Total Votes

We estimate the sharp RDD specification:

$$\text{next\_tv}_i = \beta_0 + \beta_1 \cdot \text{inc}_i + \beta_2 (R_i - c) + \beta_3 \cdot S_i (R_i - c) + \varepsilon_i$$

The coefficient $\beta_1$ on `inc` is the causal effect of incumbency on the total number of votes received in the next election.

In [5]:
# Outcome: total number of votes received in the next election
Y = df["next_tv"]

# Regressors: incumbency, plus the two RDD controls
X = df[["inc", "R_minus_c", "S_R_minus_c"]]
X = sm.add_constant(X)

# Run OLS
rdd_model = sm.OLS(Y, X, missing='drop').fit()

print(rdd_model.summary())

# Explicit values referenced in the interpretation below
print("\n===== Key values referenced in interpretation =====")
print(f"Coefficient on inc:         {rdd_model.params['inc']:.4f}")
print(f"Std error on inc:           {rdd_model.bse['inc']:.4f}")
print(f"t-statistic on inc:         {rdd_model.tvalues['inc']:.4f}")
print(f"p-value on inc:             {rdd_model.pvalues['inc']:.6f}")
print(f"Coefficient on R_minus_c:   {rdd_model.params['R_minus_c']:.4f}")
print(f"Coefficient on S_R_minus_c: {rdd_model.params['S_R_minus_c']:.4f}")
print(f"R-squared:                  {rdd_model.rsquared:.4f}")

                            OLS Regression Results                            
Dep. Variable:                next_tv   R-squared:                       0.101
Model:                            OLS   Adj. R-squared:                  0.100
Method:                 Least Squares   F-statistic:                     177.3
Date:                Thu, 30 Apr 2026   Prob (F-statistic):          6.21e-109
Time:                        21:40:53   Log-Likelihood:                -56762.
No. Observations:                4746   AIC:                         1.135e+05
Df Residuals:                    4742   BIC:                         1.136e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         7.52e+04   1522.553     49.393      